# Ben's First Pass at Working with Data

## Loading in Data

Let's make sure I can load in the data.

In [1]:
library(tidyverse)
library(tis)
library(baseballr)
library(hoopR)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘tis’


The following objects are masked from ‘package:lubridate’:

    day, hms, month, period, POSIXct, quarter, today, year, ymd


The following object is masked from ‘package:dplyr’:

    between




In [2]:
setwd('..')
getwd()
# Combine all datasets into one
#source('./code/merge-data.r')

[1] "/accounts/masters/ben_khothsombath/repos/230A-final-project"

In [3]:
od_data <- read_csv("./data/od-data.csv")
head(od_data)

Rows: 67770440 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (2): origin, destination
dbl  (2): hour, ridership
date (1): date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


date,hour,origin,destination,ridership
<date>,<dbl>,<chr>,<chr>,<dbl>
2018-01-01,0,12TH,12TH,3
2018-01-01,0,12TH,16TH,1
2018-01-01,0,12TH,BAYF,1
2018-01-01,0,12TH,CAST,3
2018-01-01,0,12TH,CIVC,2
2018-01-01,0,12TH,CONC,2


In [4]:
dim(od_data)

[1] 67770440        5

## Feature Engineering

Let's add the features that Drew's made. We'll want to ship these into a .R file and an updated .csv once we're finished.

In [5]:
print(format(object.size(od_data), units = "GB"))
od_data <- od_data |> mutate(post_covid = if_else(date > ymd("2020-03-19"), 1, 0),
                            day = case_when(wday(date) %in% seq(2, 6) ~ "weekday",
                                            wday(date) == 1 ~ "sunday",
                                            wday(date) == 7 ~ "saturday")
                        )

# Check size
print(format(object.size(od_data), units = "GB"))

[1] "2.5 Gb"
[1] "3.5 Gb"


In [6]:
head(od_data)

date,hour,origin,destination,ridership,post_covid,day
<date>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>
2018-01-01,0,12TH,12TH,3,0,weekday
2018-01-01,0,12TH,16TH,1,0,weekday
2018-01-01,0,12TH,BAYF,1,0,weekday
2018-01-01,0,12TH,CAST,3,0,weekday
2018-01-01,0,12TH,CIVC,2,0,weekday
2018-01-01,0,12TH,CONC,2,0,weekday


In [7]:
# Let's take a look at unique stations
all_stations <- unique(c(unique(od_data$origin), unique(od_data$destination)))
print(all_stations)

 [1] "12TH" "16TH" "19TH" "24TH" "ASHB" "BALB" "BAYF" "CAST" "CIVC" "COLM"
[11] "COLS" "CONC" "DALY" "DBRK" "DELN" "DUBL" "EMBR" "FRMT" "FTVL" "GLEN"
[21] "HAYW" "LAFY" "LAKE" "MCAR" "MLBR" "MONT" "NBRK" "NCON" "OAKL" "ORIN"
[31] "PHIL" "PITT" "PLZA" "POWL" "RICH" "ROCK" "SANL" "SBRN" "SFIA" "SHAY"
[41] "SSAN" "UCTY" "WARM" "WCRK" "WDUB" "WOAK" "ANTC" "PCTR" "BERY" "MLPT"


Now let's add an indicator for the line the station is at, as it might be useful

In [8]:
# BART Station Abbreviations by Line
bart_lines <- list(
  green = c("BERY", "MLPT", "WARM", "FRMT", "UCTY", "SHAY", "HAYW", "BAYF",
            "SANL", "COLS", "FTVL", "LAKE", "WOAK", "EMBR", "MONT", "POWL", 
            "CIVC", "16TH", "24TH", "GLEN", "BALB", "DALY"),
            
  red = c("RICH", "DELN", "PLZA", "NBRK", "DBRK", "ASHB", "MCAR", "19TH", 
          "12TH", "WOAK", "EMBR", "MONT", "POWL", "CIVC", "16TH", "24TH", 
          "GLEN", "BALB", "DALY", "COLM", "SSAN", "SBRN", "SFIA", "MLBR"),
          
  yellow = c("ANTC", "PCTR", "PITT", "NCON", "CONC", "PHIL", "WCRK", "LAFY", 
             "ORIN", "ROCK", "MCAR", "19TH", "12TH", "WOAK", "EMBR", "MONT", 
             "POWL", "CIVC", "16TH", "24TH", "GLEN", "BALB", "DALY", "COLM", 
             "SSAN", "SBRN", "SFIA", "MLBR"),
             
  blue = c("DUBL", "WDUB", "CAST", "BAYF", "SANL", "COLS", "FTVL", "LAKE", 
           "WOAK", "EMBR", "MONT", "POWL", "CIVC", "16TH", "24TH", "GLEN", 
           "BALB", "DALY"),
           
  orange = c("BERY", "MLPT", "WARM", "FRMT", "UCTY", "SHAY", "HAYW", "BAYF", 
             "SANL", "COLS", "FTVL", "LAKE", "12TH", "19TH", "MCAR", "ASHB", 
             "DBRK", "NBRK", "PLZA", "DELN", "RICH")
)

for (color in names(bart_lines)) {
    origin_column <- paste0("origin_", color)
    destination_column <- paste0("destination_", color)
    od_data <- od_data %>%
    mutate(
      !!origin_column := origin %in% bart_lines[[color]],
      !!destination_column := destination %in% bart_lines[[color]]
    )
}

head(od_data)

date,hour,origin,destination,ridership,post_covid,day,origin_green,destination_green,origin_red,destination_red,origin_yellow,destination_yellow,origin_blue,destination_blue,origin_orange,destination_orange
<date>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
2018-01-01,0,12TH,12TH,3,0,weekday,FALSE,FALSE,TRUE,TRUE,TRUE,TRUE,FALSE,FALSE,TRUE,TRUE
2018-01-01,0,12TH,16TH,1,0,weekday,FALSE,TRUE,TRUE,TRUE,TRUE,TRUE,FALSE,TRUE,TRUE,FALSE
2018-01-01,0,12TH,BAYF,1,0,weekday,FALSE,TRUE,TRUE,FALSE,TRUE,FALSE,FALSE,TRUE,TRUE,TRUE
2018-01-01,0,12TH,CAST,3,0,weekday,FALSE,FALSE,TRUE,FALSE,TRUE,FALSE,FALSE,TRUE,TRUE,FALSE
2018-01-01,0,12TH,CIVC,2,0,weekday,FALSE,TRUE,TRUE,TRUE,TRUE,TRUE,FALSE,TRUE,TRUE,FALSE
2018-01-01,0,12TH,CONC,2,0,weekday,FALSE,FALSE,TRUE,FALSE,TRUE,TRUE,FALSE,FALSE,TRUE,FALSE


In [9]:
# Add columns for post-covid and day-of-week

od_data <- od_data |> mutate(post_covid = if_else(date > ymd("2020-03-19"), 1, 0),
                            day = case_when(wday(date) %in% seq(2, 6) ~ "weekday",
                                            wday(date) == 1 ~ "sunday",
                                            wday(date) == 7 ~ "saturday")
                        )

head(od_data)

date,hour,origin,destination,ridership,post_covid,day,origin_green,destination_green,origin_red,destination_red,origin_yellow,destination_yellow,origin_blue,destination_blue,origin_orange,destination_orange
<date>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
2018-01-01,0,12TH,12TH,3,0,weekday,FALSE,FALSE,TRUE,TRUE,TRUE,TRUE,FALSE,FALSE,TRUE,TRUE
2018-01-01,0,12TH,16TH,1,0,weekday,FALSE,TRUE,TRUE,TRUE,TRUE,TRUE,FALSE,TRUE,TRUE,FALSE
2018-01-01,0,12TH,BAYF,1,0,weekday,FALSE,TRUE,TRUE,FALSE,TRUE,FALSE,FALSE,TRUE,TRUE,TRUE
2018-01-01,0,12TH,CAST,3,0,weekday,FALSE,FALSE,TRUE,FALSE,TRUE,FALSE,FALSE,TRUE,TRUE,FALSE
2018-01-01,0,12TH,CIVC,2,0,weekday,FALSE,TRUE,TRUE,TRUE,TRUE,TRUE,FALSE,TRUE,TRUE,FALSE
2018-01-01,0,12TH,CONC,2,0,weekday,FALSE,FALSE,TRUE,FALSE,TRUE,TRUE,FALSE,FALSE,TRUE,FALSE


In [10]:
# Holiday indicator
# Let's add an inidicator for whether a day is a holiday

holiday_dates <- as.Date(as.character(holidays(2018:2025, board = TRUE)), format = "%Y%m%d")

od_data <- od_data %>% mutate(is_holiday = date %in% holiday_dates)

In [11]:
get_home_games <- function(team_id, year) {
  mlb_schedule(year) %>%
    filter(teams_home_team_id == team_id) %>%
    pull(date) %>%
    as.Date()
}

# MLB Team IDs: Giants = 137, A's = 133
years <- 2018:2025
giants_home_dates <- map(years, ~get_home_games(137, .x)) %>% list_c()
as_home_dates <- map(years, ~get_home_games(133, .x)) %>% list_c()

#print(giants_home_dates)
# Checked a few dates against the ESPN website and it looks good for the most part.
# For robustness, you'd probably want to consider postponed/cancelled games, but it's fine for now.
#print(as_home_dates)

od_data <- od_data %>%
  mutate(
      is_giants_home = date %in% giants_home_dates,
      is_as_home = date %in% as_home_dates
  )

In [12]:
warriors_home_dates <- load_nba_schedule(2018:2025) %>%
  filter(home_id == 9) %>%
  pull(game_date) %>%
  as.Date()

#print(head(warriors_home_dates))

od_data <- od_data %>%
  mutate(
      warriors_at_coliseum = (date %in% warriors_home_dates) & (date < as.Date("2019-10-05")),
      warriors_at_chase = (date %in% warriors_home_dates) & (date >= as.Date("2019-10-05")) # Warriors first game (pre-season) at Chase
  )

head(od_data)

date,hour,origin,destination,ridership,post_covid,day,origin_green,destination_green,origin_red,⋯,destination_yellow,origin_blue,destination_blue,origin_orange,destination_orange,is_holiday,is_giants_home,is_as_home,warriors_at_coliseum,warriors_at_chase
<date>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<lgl>,<lgl>,<lgl>,⋯,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
2018-01-01,0,12TH,12TH,3,0,weekday,FALSE,FALSE,TRUE,⋯,TRUE,FALSE,FALSE,TRUE,TRUE,TRUE,FALSE,FALSE,FALSE,FALSE
2018-01-01,0,12TH,16TH,1,0,weekday,FALSE,TRUE,TRUE,⋯,TRUE,FALSE,TRUE,TRUE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE
2018-01-01,0,12TH,BAYF,1,0,weekday,FALSE,TRUE,TRUE,⋯,FALSE,FALSE,TRUE,TRUE,TRUE,TRUE,FALSE,FALSE,FALSE,FALSE
2018-01-01,0,12TH,CAST,3,0,weekday,FALSE,FALSE,TRUE,⋯,FALSE,FALSE,TRUE,TRUE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE
2018-01-01,0,12TH,CIVC,2,0,weekday,FALSE,TRUE,TRUE,⋯,TRUE,FALSE,TRUE,TRUE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE
2018-01-01,0,12TH,CONC,2,0,weekday,FALSE,FALSE,TRUE,⋯,TRUE,FALSE,FALSE,TRUE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE


Let's test the function for filtering the data. I'll look at Drew's code.

In [13]:
#od_complete <- od_data |>
#    complete(date, hour, origin, destination, fill = list(ridership = 0))

#format(object.size(dat_complete), units = "GB")

ERROR: Error: object 'dat_complete' not found


In [ ]:
#head(od_complete)

Okay maybe let's filter before performing feature engineering...